# Processes

---

## Process

A process is an abstraction describing the state and execution of a currently running program. This includes the resources (memory, CPU time) that the operating system dedicates to running the program.

Each process has its own memory image, which contains program data, instructions, global/static variables, heap memory, shared memory, stack memory, command line arguments and the environment.

Each process is given an ID by the OS, which can be obtained with the `getpid` function.

---

## Memory

System memory is divided into two spaces:

### Kernel

Contains the memory content and instructions of the operating system.
- Only processes with certain priveleges can gain read/write access
- Stores process IDs
- Hardware can only be accessed by going through this layer, for protection purposes

### User

Contains the states, data and instructions to run a program.
- Priveleges depend on the creator and can be modified by the user
- Different processes can be run simultaneously and independent of each other
- A parent can execute multiple child processes

User space processes use system calls to access kernel space.

---

## Exec

Similar to a GOTO, but for processes.
```c
int execl(const char* path, const char* arg, ..., NULL);
```
If an `execl` call is successful, the entire process memory space will be replaced by the program that we want to switch to. It does not start a new process. Once the process is finished, it does not go back to our original program.
```c
// NULL is the last arg because command line arguments need to be null-terminated
execl("/usr/bin/echo", "echo", "Hello World!", NULL);

// Program should only reach this point if execl is unsuccessful
perror("Cannot execute");
exit(1);
```
If unsuccessful, we go back to the original program.

---

## Fork

Creates a child process which is a copy of the current memory image of the parent (with a new PID), and then executes both in parallel.
```c
pid_t fork(void);
```
The child gets a copy of the parent's stack, heap and data space. Pointers and variables are only copied-on-write to the new memory space.

The `fork` is called once, but returns different values depending on which process you are in:
- Returns PID of the child, if in parent
- Returns 0, if in child
- A negative value, if an error occured
```c
pid_t pid = fork();

if (pid == 0) {
    // In child process
    return 0;
}
if (pid < 0) {
    perror("Cannot fork");
    return 1;
}

// In parent process
```
The running order of the parent and child is non-deterministic. If the parent finishes executing before the child, there will be a zombie process.

---

## Wait

To implictly instruct the OS to wait for the child process to terminate, use `wait`/`waitpid`.
```c
pid_t wait(int* status);
pid_t waitpid(pid_t pid, int* status, int options);
```
Returns the PID of the child. The state at which the child terminates (successful or not) is set in `status`.

`wait` is important to clean up the process from the process table. When your child process terminates, it is the parent process's responsibility to clean it up, otherwise it becomes a zombie.

Generally `wait` is a blocking operation, though there is a flag that can be used to make it non-blocking.

---

## Race Conditions

Since the order of execution of processes (or threads) is unknown, race conditions can arise. If two processes try to modify the same shared resource at the same time, we can get undefined behaviour.

Because the program sometimes behaves correctly, and sometimes doesn't, race conditions are extremely hard to detect and debug. The following are some ways to prevent ourselves from creating them:

### Data Related

The most common type of race condition, where two processes try to modify the same piece of data at the same time.
- Make your code/function pure; make duplicates of data that you are changing and that are in use by other processes
- Make the parallel processes access seperate regions of the data
- Use different forms of locks, such as mutexes

### Event Related

Conditions that are to do with events that happen in the wrong order. For example, you receive a signal before you've set up your signal handler, or you `pause` after you've received a signal, or you read from shared memory before valid data has been written to it.

These are harder to prevent. If you do not want the program to be interrupted for some duration, consider masking and queueing signals. If an event requires waiting for something that has already occured, set some flag to check prior to beginning that event. Make sure that the precautions taken are atomic.